In [5]:
#Importar librerias y ademas encontrar el pdf por el directorio

from pathlib import Path 
import re 
from pypdf import PdfReader
import pandas as pd 

ruta_pdf =Path("../data/raw/normativa/Gaceta36610 Resolucion GEE 500 18-07-2024.pdf")

print("Ruta completa: ", ruta_pdf.resolve())
print("existe: ",ruta_pdf.exists())

lector = PdfReader(ruta_pdf)
print("Paginas: ",len(lector.pages))




Ruta completa:  C:\Users\gaand\Proyecto-BD2\Proyectos-BD2\data\raw\normativa\Gaceta36610 Resolucion GEE 500 18-07-2024.pdf
existe:  True
Paginas:  8


In [6]:
paginas = []
for numero, pagina in enumerate(lector.pages,start = 1):
    texto = pagina.extract_text() or ""
    paginas.append((numero,texto))

for numero,texto in paginas:
    print(f"===== PAGINA {numero} ({len(texto)}) caracteres) =====")
    print(texto)
    print()

===== PAGINA 1 (3725) caracteres) =====
1
La Gaceta   
B.
Sección  B  Avisos Legales
REPÚBLICA   DE   HONDURAS - TEGUCIGALPA, M. D. C.,  12  DE AGOSTO  DEL   2024     No. 36,610
Sección “B”
CERTIFICACIÓN
El infrascrito, Secretario General de la Comisión Nacional 
de Bancos y Seguros CERTIFICA la parte conducente 
del Acta de la Sesión No.1814 celebrada en Tegucigalpa, 
Municipio del Distrito Central el dieciocho de julio de dos 
mil veinticuatro, con la asistencia de los Comisionados 
MARCIO GIOVANNY SIERRA DISCUA, Presidente; 
ALBA LUZ VALLADARES OCONNOR, Comisionada 
Propietaria; ESDRAS JOSIEL SÁNCHEZ BARAHONA, 
Comisionado Propietario; JUAN MANUEL SIBAJA 
SALINAS, Secretario General; que dice: 
“… 4. Asuntos de la Gerencia de Estudios Económicos, 
Regulación, Competencia e Innovación Financiera:  … 
literal e) …  RESOLUCIÓN GEE No.500/18-07-2024.- La 
Comisión Nacional de Bancos y Seguros, 
CONSIDERANDO (1):  Que el Artículo 1 de la Ley de 
la Comisión Nacional de Bancos y Seguros e

In [7]:
texto_completo = "\n".join(texto for numero, texto in paginas)
print("caracteres totales:", len(texto_completo))

caracteres totales: 27315


In [8]:
inicio = texto_completo.find("ARTÍCULO 1.-")
fin = texto_completo.find("2. Comunicar")

cuerpo = texto_completo[inicio:fin]


print("inicio: ", inicio, "| fin:",fin)
print("caracteres del cuerpo", len(cuerpo),)
print()
print("--- Inicio del cuerpo ---")
print(cuerpo[:200])
print()
print("--- Inicio de la parte instituciones ---")
print(cuerpo[-200:])

inicio:  4802 | fin: 26180
caracteres del cuerpo 21378

--- Inicio del cuerpo ---
ARTÍCULO 1.- OBJETO
Los presentes lineamientos tienen por objeto establecer 
los controles mínimos de seguridad que deben aplicar 
las Instituciones Supervisadas por la Comisión Nacional 
de Bancos y 

--- Inicio de la parte instituciones ---
suario financiero.
ARTÍCULO 21.- VIGENCIA 
Lo descrito en los presentes Lineamientos no tiene efecto 
retroactivo y entrarán en vigencia a partir de su publicación 
en el Diario Oficial “La Gaceta”. 



In [9]:
# Cortar cada sección en trozos, justo antes de cada número de punto (1., 2., ... 10.)
puntos = re.split(r"\n(?=ARTÍCULO \d+\.-)", cuerpo)

print("trozos :", len(puntos))
print()

# Ver cómo empieza cada trozo de la sección usuario
for i, trozo in enumerate(puntos):
    print(i, "->", trozo[:45].replace("\n", " "))

trozos : 21

0 -> ARTÍCULO 1.- OBJETO Los presentes lineamiento
1 -> ARTÍCULO 2.- ALCANCE Las disposiciones de los
2 -> ARTÍCULO 3.- DEFINICIONES Para los efectos de
3 -> ARTÍCULO 4.- DEL SISTEMA DE MONITOREO DE  OPE
4 -> ARTÍCULO 5.- DE LA NOTIFICACIÓN AL  TARJETAHA
5 -> ARTÍCULO 6.- GESTIÓN DE RIESGOS  RELACIONADOS
6 -> ARTÍCULO 7.- RESPONSABILIDADES DE LAS  INSTIT
7 -> ARTÍCULO 8.- AUTENTICACIÓN DEL  TARJETAHABIEN
8 -> ARTÍCULO 9.- ENROLAMIENTO DE TARJETAS  CON SE
9 -> ARTÍCULO 10.- SERVICIO DE ATENCIÓN AL  TARJET
10 -> ARTÍCULO 11.- PROCEDIMIENTO PARA REPORTAR  CA
11 -> ARTÍCULO 12.- PROCESO DE RECLAMO  SOBRE TRANS
12 -> ARTÍCULO 13.- INVESTIGACIÓN DE RECLAMOS  POR 
13 -> ARTÍCULO 14.- RESPONSABILIDADES DE LAS  FUNCI
14 -> ARTÍCULO 15.- PROGRAMAS DE EDUCACIÓN  FINANCI
15 -> ARTÍCULO 16.- RESPONSABILIDAD DE LAS   INSTIT
16 -> ARTÍCULO 17.- RESPONSABILIDAD DE LOS  TARJETA
17 -> ARTÍCULO 18.- CASOS NO PREVISTOS  Los casos n
18 -> ARTÍCULO 19.- SANCIONES  El incumplimiento al
19 -> ART

In [10]:
ruido = [
    r"^\s*La Gaceta\s*$",
    r"^\s*B\.\s*$",
    r"^\s*Sección\s+B\s+Avisos\s+Legales\s*$",
    r"^\s*REPÚBLICA\s+DE\s+HONDURAS.*No\.\s*36,610\s*$",
    r"^\s*\d{1,2}\s*$",
]

def limpiar_trozo(trozo):
    lineas = []
    for linea in trozo.split("\n"):
        if any(re.match(p, linea) for p in ruido):
            continue
        lineas.append(linea)
    return re.sub(r"\s+", " ", " ".join(lineas)).strip()

limpios = [limpiar_trozo(t) for t in puntos]

for i, t in enumerate(limpios):
    print(i, "|", len(t), "|", t[:70])

print()
print("quedan encabezados?:", any("Avisos Legales" in t or "No. 36,610" in t for t in limpios))

0 | 524 | ARTÍCULO 1.- OBJETO Los presentes lineamientos tienen por objeto estab
1 | 235 | ARTÍCULO 2.- ALCANCE Las disposiciones de los presentes lineamientos s
2 | 7486 | ARTÍCULO 3.- DEFINICIONES Para los efectos de los presentes lineamient
3 | 1006 | ARTÍCULO 4.- DEL SISTEMA DE MONITOREO DE OPERACIONES CON TARJETAS Las 
4 | 1033 | ARTÍCULO 5.- DE LA NOTIFICACIÓN AL TARJETAHABIENTE DE LAS OPERACIONES 
5 | 614 | ARTÍCULO 6.- GESTIÓN DE RIESGOS RELACIONADOS CON MODALIDADES DE FRAUDE
6 | 926 | ARTÍCULO 7.- RESPONSABILIDADES DE LAS INSTITUCIONES QUE BRINDEN SERVIC
7 | 945 | ARTÍCULO 8.- AUTENTICACIÓN DEL TARJETAHABIENTE PARA OPERACIONES NO PRE
8 | 297 | ARTÍCULO 9.- ENROLAMIENTO DE TARJETAS CON SERVICIOS DE PAGO MÓVIL Las 
9 | 361 | ARTÍCULO 10.- SERVICIO DE ATENCIÓN AL TARJETAHABIENTE PARA CASOS DE OP
10 | 597 | ARTÍCULO 11.- PROCEDIMIENTO PARA REPORTAR CASOS DE OPERACIONES NO RECO
11 | 968 | ARTÍCULO 12.- PROCESO DE RECLAMO SOBRE TRANSACCIONES MONETARIAS NO REC
12 | 689 | ARTÍCULO 13.

In [11]:
capitulo = "CAPÍTULO I DISPOSICIONES GENERALES"
articulos = []

for t in limpios:
    seccion = capitulo
    m = re.search(r"\s*(CAPÍTULO [IVX]+\s+.*)$", t)
    if m:
        t = t[:m.start()]
        capitulo = m.group(1)
    numero = int(re.match(r"ARTÍCULO (\d+)", t).group(1))
    t = re.sub(r"^ARTÍCULO \d+\.-\s*", "", t)
    articulos.append({"numero": numero, "seccion": seccion, "texto": t})

for a in articulos:
    print(a["numero"], "|", a["seccion"][:35], "|", len(a["texto"]))

1 | CAPÍTULO I DISPOSICIONES GENERALES | 511
2 | CAPÍTULO I DISPOSICIONES GENERALES | 222
3 | CAPÍTULO I DISPOSICIONES GENERALES | 7406
4 | CAPÍTULO II DE LOS CONTROLES MÍNIMO | 993
5 | CAPÍTULO II DE LOS CONTROLES MÍNIMO | 1020
6 | CAPÍTULO II DE LOS CONTROLES MÍNIMO | 601
7 | CAPÍTULO II DE LOS CONTROLES MÍNIMO | 913
8 | CAPÍTULO II DE LOS CONTROLES MÍNIMO | 932
9 | CAPÍTULO II DE LOS CONTROLES MÍNIMO | 284
10 | CAPÍTULO II DE LOS CONTROLES MÍNIMO | 347
11 | CAPÍTULO II DE LOS CONTROLES MÍNIMO | 583
12 | CAPÍTULO II DE LOS CONTROLES MÍNIMO | 954
13 | CAPÍTULO II DE LOS CONTROLES MÍNIMO | 675
14 | CAPÍTULO II DE LOS CONTROLES MÍNIMO | 432
15 | CAPÍTULO II DE LOS CONTROLES MÍNIMO | 868
16 | CAPÍTULO III DE LAS DISPOSICIONES F | 1510
17 | CAPÍTULO III DE LAS DISPOSICIONES F | 191
18 | CAPÍTULO III DE LAS DISPOSICIONES F | 155
19 | CAPÍTULO III DE LAS DISPOSICIONES F | 278
20 | CAPÍTULO III DE LAS DISPOSICIONES F | 750
21 | CAPÍTULO III DE LAS DISPOSICIONES F | 162


In [12]:
def trocear(texto, maximo=400):
    frases = re.split(r"(?<=[.;:])\s+", texto)
    piezas, actual = [], ""
    for f in frases:
        if actual and len(actual) + len(f) + 1 > maximo:
            piezas.append(actual)
            actual = f
        else:
            actual = (actual + " " + f).strip()
    if actual:
        piezas.append(actual)
    return piezas

In [13]:
prefijo = "Lineamientos para la prevención de operaciones no reconocidas con tarjetas, artículo"
registros = []

for a in articulos:
    if a["numero"] == 3:
        intro, resto = a["texto"].split("definiciones:", 1)
        piezas = re.split(r"\s(?=\d{1,2}\.\s+[A-ZÁÉÍÓÚÑ])", resto.strip())
    else:
        piezas = trocear(a["texto"])
    for p in piezas:
        registros.append({
            "documento": ruta_pdf.stem,
            "categoria": ruta_pdf.parent.name,
            "seccion": a["seccion"],
            "numero_item": a["numero"],
            "contenido": f"{prefijo} {a['numero']}: {p}",
        })

print("total de registros:", len(registros))

total de registros: 69


In [14]:
# Metadatos que faltan
for i, r in enumerate(registros):
    r["fuente"] = "CNBS - Resolución GEE No.500/18-07-2024 (La Gaceta 36,610)"
    r["chunk_index"] = i

df = pd.DataFrame(registros)

# Mismo orden de columnas que el decálogo
df = df[["documento", "categoria", "fuente", "seccion", "numero_item", "chunk_index", "contenido"]]

# Guardar
salida = Path("../data/processed/chunks_gaceta_gee500.csv")
df.to_csv(salida, index=False, encoding="utf-8")

print("guardado en:", salida.resolve())
print("filas:", len(df))
df.head()

guardado en: C:\Users\gaand\Proyecto-BD2\Proyectos-BD2\data\processed\chunks_gaceta_gee500.csv
filas: 69


,documento,categoria,fuente,seccion,numero_item,chunk_index,contenido
0,Gaceta36610 Resolucion GEE 500 18-07-2024,normativa,CNBS - Resolución GEE No.500/18-07-2024 (La Ga...,CAPÍTULO I DISPOSICIONES GENERALES,1,0,Lineamientos para la prevención de operaciones...
1,Gaceta36610 Resolucion GEE 500 18-07-2024,normativa,CNBS - Resolución GEE No.500/18-07-2024 (La Ga...,CAPÍTULO I DISPOSICIONES GENERALES,1,1,Lineamientos para la prevención de operaciones...
2,Gaceta36610 Resolucion GEE 500 18-07-2024,normativa,CNBS - Resolución GEE No.500/18-07-2024 (La Ga...,CAPÍTULO I DISPOSICIONES GENERALES,2,2,Lineamientos para la prevención de operaciones...
3,Gaceta36610 Resolucion GEE 500 18-07-2024,normativa,CNBS - Resolución GEE No.500/18-07-2024 (La Ga...,CAPÍTULO I DISPOSICIONES GENERALES,3,3,Lineamientos para la prevención de operaciones...
4,Gaceta36610 Resolucion GEE 500 18-07-2024,normativa,CNBS - Resolución GEE No.500/18-07-2024 (La Ga...,CAPÍTULO I DISPOSICIONES GENERALES,3,4,Lineamientos para la prevención de operaciones...
